## CNN

## 2-class 레이블 정의
- AD → label 1
- Normal → label 0

## 데이터 로딩 코드 (2-class)

In [ ]:
import numpy as np
import torch    
import scipy.io as sio

all_X = []
all_y = []

# 1) AD
mat_ad = sio.loadmat("AD.mat", struct_as_record=False, squeeze_me=True)
data_ad = mat_ad['AD']
rec_ad = data_ad[0]
arr_uv_ad = rec_ad.epoch
arr_v_ad = arr_uv_ad * 1e-6
X_ad = arr_v_ad.transpose(2, 0, 1)  # (epochs, channels, time)
all_X.append(X_ad)
all_y.extend([1]*X_ad.shape[0])     # AD = 1

# 2) Normal
mat_nor = sio.loadmat("Normal.mat", struct_as_record=False, squeeze_me=True)
data_nor = mat_nor['normal']
rec_nor = data_nor[0]
arr_uv_nor = rec_nor.epoch
arr_v_nor = arr_uv_nor * 1e-6
X_nor = arr_v_nor.transpose(2, 0, 1)
all_X.append(X_nor)
all_y.extend([0]*X_nor.shape[0])    # Normal = 0

# 합치기
X_total = np.concatenate(all_X, axis=0)
y_total = np.array(all_y)

print("데이터 shape:", X_total.shape)
print("레이블 shape:", y_total.shape)
print("레이블 분포:", np.bincount(y_total))


데이터 shape: (178, 4, 600)
레이블 shape: (178,)
레이블 분포: [119  59]


## PyTorch Tensor 변환

In [2]:
X_tensor = torch.tensor(X_total, dtype=torch.float32).unsqueeze(1)
y_tensor = torch.tensor(y_total, dtype=torch.long)

print(X_tensor.shape)  # (samples, 1, channels, time)


torch.Size([178, 1, 4, 600])


## CNN 모델 (2-class)
 - 마지막 출력만 2개로 설정

In [4]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

dataset = TensorDataset(X_tensor, y_tensor)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

class EEG_CNN(nn.Module):
    def __init__(self):
        super(EEG_CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(32 * (4//4) * (600//4), 128)
        self.fc2 = nn.Linear(128, 2)  # 2-class

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = EEG_CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


## 학습 루프

In [5]:
for epoch in range(20):
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")


Epoch 1, Loss: 0.4063
Epoch 2, Loss: 0.7387
Epoch 3, Loss: 0.4375
Epoch 4, Loss: 0.7834
Epoch 5, Loss: 0.7147
Epoch 6, Loss: 1.2166
Epoch 7, Loss: 0.5698
Epoch 8, Loss: 0.3623
Epoch 9, Loss: 0.3783
Epoch 10, Loss: 0.4377
Epoch 11, Loss: 0.7775
Epoch 12, Loss: 0.7456
Epoch 13, Loss: 1.0701
Epoch 14, Loss: 0.7240
Epoch 15, Loss: 0.4544
Epoch 16, Loss: 0.3534
Epoch 17, Loss: 0.7618
Epoch 18, Loss: 0.4580
Epoch 19, Loss: 0.3534
Epoch 20, Loss: 0.7533


## SVM + 교차검증 : 머신러닝

In [6]:
import numpy as np
import scipy.io as sio
import pywt
from scipy.signal import coherence
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix
from itertools import combinations

# ---------------------------
# 1) 데이터 로드
# ---------------------------
mat_ad = sio.loadmat("AD.mat", struct_as_record=False, squeeze_me=True)
data_ad = mat_ad['AD']
rec_ad = data_ad[0]
arr_uv_ad = rec_ad.epoch
arr_v_ad = arr_uv_ad * 1e-6
X_ad = arr_v_ad.transpose(2, 0, 1)

mat_nor = sio.loadmat("Normal.mat", struct_as_record=False, squeeze_me=True)
data_nor = mat_nor['normal']
rec_nor = data_nor[0]
arr_uv_nor = rec_nor.epoch
arr_v_nor = arr_uv_nor * 1e-6
X_nor = arr_v_nor.transpose(2, 0, 1)

X_total = np.concatenate([X_ad, X_nor], axis=0)
y_total = np.concatenate([
    np.ones(X_ad.shape[0]),
    np.zeros(X_nor.shape[0])
])

print(f"원본 데이터 shape: {X_total.shape}")

# ---------------------------
# 2) 웨이블릿 특징 추출
# ---------------------------
def extract_wavelet_features(X, wavelet='db4', level=4):
    n_samples, n_channels, n_times = X.shape
    features = np.zeros((n_samples, n_channels * level))
    for i in range(n_samples):
        for ch in range(n_channels):
            coeffs = pywt.wavedec(X[i, ch, :], wavelet, level=level)
            energies = [np.sum(c**2) for c in coeffs[1:]]
            features[i, ch*level:(ch+1)*level] = energies
    return features

wavelet_features = extract_wavelet_features(X_total, wavelet='db4', level=4)
print(f"웨이블릿 feature shape: {wavelet_features.shape}")

# ---------------------------
# 3) coherence feature 추출
# ---------------------------
def extract_coherence_features(X, sfreq=200):
    """
    X shape: (n_samples, n_channels, n_times)
    """
    n_samples, n_channels, n_times = X.shape
    ch_pairs = list(combinations(range(n_channels), 2))
    features = np.zeros((n_samples, len(ch_pairs)))
    
    for i in range(n_samples):
        values = []
        for ch1, ch2 in ch_pairs:
            f, Cxy = coherence(X[i, ch1, :], X[i, ch2, :], fs=sfreq, nperseg=sfreq*2)
            mean_coh = np.mean(Cxy[(f>=8) & (f<=13)])  # alpha-band coherence
            values.append(mean_coh)
        features[i, :] = values
    return features

coherence_features = extract_coherence_features(X_total)
print(f"coherence feature shape: {coherence_features.shape}")

# ---------------------------
# 4) 웨이블릿 + coherence 결합
# ---------------------------
combined_features = np.concatenate([wavelet_features, coherence_features], axis=1)
print(f"combined feature shape: {combined_features.shape}")

# ---------------------------
# 5) SMOTE
# ---------------------------
smote = SMOTE(random_state=42)
features_resampled, y_resampled = smote.fit_resample(combined_features, y_total)
print("SMOTE 후 shape:", features_resampled.shape)
print("클래스 분포:", np.bincount(y_resampled.astype(int)))

# ---------------------------
# 6) 표준화
# ---------------------------
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_resampled)

# ---------------------------
# 7) SVM + 교차검증
# ---------------------------
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(features_scaled, y_resampled)):
    X_train, X_val = features_scaled[train_idx], features_scaled[val_idx]
    y_train, y_val = y_resampled[train_idx], y_resampled[val_idx]

    clf = SVC(kernel='rbf', C=1, gamma='scale', class_weight='balanced', random_state=42)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    cm = confusion_matrix(y_val, y_pred)
    
    print(f"\n--- Fold {fold+1} ---")
    print(f"Fold {fold+1} Test Accuracy: {acc:.4f}")
    print(f"Confusion Matrix:\n{cm}")
    results.append(acc)

print(f"\nAverage 5-Fold Accuracy: {np.mean(results):.4f}")



원본 데이터 shape: (178, 4, 600)
웨이블릿 feature shape: (178, 16)
coherence feature shape: (178, 6)
combined feature shape: (178, 22)
SMOTE 후 shape: (238, 22)
클래스 분포: [119 119]

--- Fold 1 ---
Fold 1 Test Accuracy: 0.9792
Confusion Matrix:
[[23  1]
 [ 0 24]]

--- Fold 2 ---
Fold 2 Test Accuracy: 0.9167
Confusion Matrix:
[[20  4]
 [ 0 24]]

--- Fold 3 ---
Fold 3 Test Accuracy: 0.9375
Confusion Matrix:
[[21  3]
 [ 0 24]]

--- Fold 4 ---
Fold 4 Test Accuracy: 0.9149
Confusion Matrix:
[[23  0]
 [ 4 20]]

--- Fold 5 ---
Fold 5 Test Accuracy: 0.9787
Confusion Matrix:
[[23  1]
 [ 0 23]]

Average 5-Fold Accuracy: 0.9454


In [7]:
import numpy as np
import scipy.io as sio
import pywt
from scipy.signal import coherence
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix
from itertools import combinations

# ---------------------------
# 1) 데이터 로드
# ---------------------------
mat_ad = sio.loadmat("AD.mat", struct_as_record=False, squeeze_me=True)
data_ad = mat_ad['AD']
rec_ad = data_ad[0]
arr_uv_ad = rec_ad.epoch
arr_v_ad = arr_uv_ad * 1e-6
X_ad = arr_v_ad.transpose(2, 0, 1)

mat_nor = sio.loadmat("Normal.mat", struct_as_record=False, squeeze_me=True)
data_nor = mat_nor['normal']
rec_nor = data_nor[0]
arr_uv_nor = rec_nor.epoch
arr_v_nor = arr_uv_nor * 1e-6
X_nor = arr_v_nor.transpose(2, 0, 1)

X_total = np.concatenate([X_ad, X_nor], axis=0)
y_total = np.concatenate([
    np.ones(X_ad.shape[0]),
    np.zeros(X_nor.shape[0])
])

print(f"원본 데이터 shape: {X_total.shape}")

# ---------------------------
# 2) 웨이블릿 특징
# ---------------------------
def extract_wavelet_features(X, wavelet='db4', level=4):
    n_samples, n_channels, n_times = X.shape
    features = np.zeros((n_samples, n_channels * level))
    for i in range(n_samples):
        for ch in range(n_channels):
            coeffs = pywt.wavedec(X[i, ch, :], wavelet, level=level)
            energies = [np.sum(c**2) for c in coeffs[1:]]
            features[i, ch*level:(ch+1)*level] = energies
    return features

wavelet_features = extract_wavelet_features(X_total)

# ---------------------------
# 3) coherence 특징
# ---------------------------
def extract_coherence_features(X, sfreq=200):
    n_samples, n_channels, n_times = X.shape
    ch_pairs = list(combinations(range(n_channels), 2))
    features = np.zeros((n_samples, len(ch_pairs)))
    
    for i in range(n_samples):
        values = []
        for ch1, ch2 in ch_pairs:
            f, Cxy = coherence(X[i, ch1, :], X[i, ch2, :], fs=sfreq, nperseg=sfreq*2)
            mean_coh = np.mean(Cxy[(f>=8) & (f<=13)])  # alpha-band
            values.append(mean_coh)
        features[i, :] = values
    return features

coherence_features = extract_coherence_features(X_total)

# ---------------------------
# 4) 피처 결합
# ---------------------------
combined_features = np.concatenate([wavelet_features, coherence_features], axis=1)
print(f"combined feature shape: {combined_features.shape}")

# ---------------------------
# 5) SMOTE
# ---------------------------
smote = SMOTE(random_state=42)
features_resampled, y_resampled = smote.fit_resample(combined_features, y_total)
print(f"SMOTE 후 shape: {features_resampled.shape}")
print("클래스 분포:", np.bincount(y_resampled.astype(int)))

# ---------------------------
# 6) 표준화
# ---------------------------
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features_resampled)

# ---------------------------
# 7) SVM + GridSearchCV
# ---------------------------
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 0.01, 0.1, 1],
    'kernel': ['rbf']
}

svm = SVC(class_weight='balanced', random_state=42)
grid = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    verbose=2,
    n_jobs=-1
)

grid.fit(features_scaled, y_resampled)

print("\nBest Parameters:", grid.best_params_)
print(f"Best Cross-Validation Accuracy: {grid.best_score_:.4f}")


원본 데이터 shape: (178, 4, 600)
combined feature shape: (178, 22)
SMOTE 후 shape: (238, 22)
클래스 분포: [119 119]
Fitting 5 folds for each of 16 candidates, totalling 80 fits

Best Parameters: {'C': 10, 'gamma': 0.01, 'kernel': 'rbf'}
Best Cross-Validation Accuracy: 0.9875


## CNN 모델

In [8]:
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# ---------------------------
# 1) 데이터 로드
# ---------------------------
mat_ad = sio.loadmat("AD.mat", struct_as_record=False, squeeze_me=True)
data_ad = mat_ad['AD']
rec_ad = data_ad[0]
arr_uv_ad = rec_ad.epoch
arr_v_ad = arr_uv_ad * 1e-6
X_ad = arr_v_ad.transpose(2, 0, 1)

mat_nor = sio.loadmat("Normal.mat", struct_as_record=False, squeeze_me=True)
data_nor = mat_nor['normal']
rec_nor = data_nor[0]
arr_uv_nor = rec_nor.epoch
arr_v_nor = arr_uv_nor * 1e-6
X_nor = arr_v_nor.transpose(2, 0, 1)

# ---------------------------
# 2) 데이터 합치기
# ---------------------------
X_total = np.concatenate([X_ad, X_nor], axis=0)
y_total = np.concatenate([
    np.ones(X_ad.shape[0]),
    np.zeros(X_nor.shape[0])
])

print(f"원본 데이터 shape: {X_total.shape}")   # (n_samples, n_channels, n_times)

# ---------------------------
# 3) train/val split
# ---------------------------
X_train, X_val, y_train, y_val = train_test_split(
    X_total, y_total, test_size=0.2, random_state=42, stratify=y_total
)

# torch tensor
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)

# CNN은 (N, C, T) 를 (N, 1, C, T) 로 reshape
X_train_tensor = X_train_tensor.unsqueeze(1)
X_val_tensor = X_val_tensor.unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=16, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=16, shuffle=False)

# ---------------------------
# 4) CNN 모델 정의
# ---------------------------
class EEG_CNN(nn.Module):
    def __init__(self):
        super(EEG_CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=(4,5), padding=(0,2))  # 4채널 합치며 5타임 윈도우
        self.pool = nn.MaxPool2d((1,2))
        self.conv2 = nn.Conv2d(16, 32, kernel_size=(1,5), padding=(0,2))
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(32 * 1 * (600//4), 64)
        self.fc2 = nn.Linear(64, 2)
    
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.dropout(torch.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

model = EEG_CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ---------------------------
# 5) 학습 루프
# ---------------------------
n_epochs = 20
for epoch in range(n_epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0
    
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * X_batch.size(0)
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)
        
    train_acc = correct / total
    train_loss = running_loss / total

    # validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss += loss.item() * X_batch.size(0)
            preds = torch.argmax(outputs, dim=1)
            val_correct += (preds == y_batch).sum().item()
            val_total += y_batch.size(0)
    val_acc = val_correct / val_total
    val_loss = val_loss / val_total
    
    print(f"Epoch {epoch+1:2d} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

# ---------------------------
# 6) 최종 평가
# ---------------------------
model.eval()
with torch.no_grad():
    outputs = model(X_val_tensor)
    preds = torch.argmax(outputs, dim=1)
    acc = accuracy_score(y_val, preds.numpy())
    cm = confusion_matrix(y_val, preds.numpy())

print(f"\n최종 Test Accuracy: {acc:.4f}")
print("Confusion Matrix:\n", cm)


원본 데이터 shape: (178, 4, 600)
Epoch  1 | Train Acc: 0.6620 | Val Acc: 0.6667
Epoch  2 | Train Acc: 0.6549 | Val Acc: 0.6667
Epoch  3 | Train Acc: 0.6620 | Val Acc: 0.6667
Epoch  4 | Train Acc: 0.6690 | Val Acc: 0.6667
Epoch  5 | Train Acc: 0.6690 | Val Acc: 0.6667
Epoch  6 | Train Acc: 0.6690 | Val Acc: 0.6667
Epoch  7 | Train Acc: 0.6690 | Val Acc: 0.6667
Epoch  8 | Train Acc: 0.6690 | Val Acc: 0.6667
Epoch  9 | Train Acc: 0.6690 | Val Acc: 0.6667
Epoch 10 | Train Acc: 0.6690 | Val Acc: 0.6667
Epoch 11 | Train Acc: 0.6690 | Val Acc: 0.6667
Epoch 12 | Train Acc: 0.6690 | Val Acc: 0.6667
Epoch 13 | Train Acc: 0.6690 | Val Acc: 0.6667
Epoch 14 | Train Acc: 0.6690 | Val Acc: 0.6667
Epoch 15 | Train Acc: 0.6690 | Val Acc: 0.6667
Epoch 16 | Train Acc: 0.6690 | Val Acc: 0.6667
Epoch 17 | Train Acc: 0.6690 | Val Acc: 0.6667
Epoch 18 | Train Acc: 0.6690 | Val Acc: 0.6667
Epoch 19 | Train Acc: 0.6690 | Val Acc: 0.6667
Epoch 20 | Train Acc: 0.6690 | Val Acc: 0.6667

최종 Test Accuracy: 0.6667
Confus

In [10]:
import nbformat

with open("AD_modeling.ipynb", "r", encoding="utf-8") as f:
    try:
        nb = nbformat.read(f, as_version=4)
        print("OK")
    except Exception as e:
        print("오류:", e)


오류: Notebook does not appear to be JSON: ''...
